In [1]:
"""YOLO26x object detection trên tối đa 2 GPU Kaggle.

Tên file đầu vào:
    l29-v001_s000001_f000000048.jpg

JSON đầu ra:
    [
      {
        "video_id": "l29-v001",
        "shot_id": 1,
        "frames_id": 48,
        "objects": {"person": 6, "boat": 1}
      }
    ]

File kết quả:
    /kaggle/working/yolo_object_results_yolo26x.json
"""

from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path


# =============================================================================
# CẤU HÌNH (ĐÃ TỐI ƯU HÓA TỐC ĐỘ)
# =============================================================================

MODEL_NAME = "yolo26l.pt"   # YOLO26x pretrained COCO-80
CONFIDENCE = 0.40           # có thể tinh chỉnh trên validation set
IMAGE_SIZE = 736            # Đã giảm xuống 640 để tăng tốc độ xử lý FPS
BATCH_SIZE = 64            # Đã tăng lên 32 để tận dụng tối đa VRAM T4
MAX_GPUS = 2                # tự dùng GPU 0 và 1 nếu Kaggle cung cấp T4 x2
MODEL_TAG = Path(MODEL_NAME).stem

KAGGLE_INPUT = Path("/kaggle/input/datasets/trieu241007/keyframes-p1/")
EXTRACT_DIR = Path("/kaggle/working/keyframes")
OUTPUT_JSON = Path(f"/kaggle/working/yolo_object_results_{MODEL_TAG}.json")
CHECKPOINT_PREFIX = Path(
    f"/kaggle/working/yolo_object_checkpoint_{MODEL_TAG}"
)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
FILENAME_PATTERN = re.compile(
    r"^(?P<video_id>l\d+-v\d+)_s(?P<shot_id>\d+)_f(?P<frames_id>\d+)$",
    flags=re.IGNORECASE,
)


# =============================================================================
# CÀI PACKAGE
# =============================================================================

# YOLO26 cần Ultralytics mới; -U cũng nâng cấp bản cũ có sẵn trên Kaggle.
print("Đang kiểm tra/cập nhật Ultralytics...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"],
    check=True,
)

import torch
from tqdm.auto import tqdm
from ultralytics import YOLO


# =============================================================================
# TÌM HOẶC GIẢI NÉN KEYFRAMES
# =============================================================================

def contains_images(directory: Path) -> bool:
    return directory.is_dir() and any(
        path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        for path in directory.rglob("*")
    )


def find_keyframes_dir() -> Path:
    """Tìm folder keyframes trong Kaggle Input hoặc giải nén từ file TAR."""
    # Trường hợp upload trực tiếp folder keyframes.
    for candidate in sorted(KAGGLE_INPUT.glob("**/keyframes")):
        if contains_images(candidate):
            print(f"Tìm thấy keyframes: {candidate}")
            return candidate

    # Trường hợp upload một file TAR chứa folder keyframes.
    tar_files = sorted(KAGGLE_INPUT.glob("**/*.tar"))
    if not tar_files:
        raise FileNotFoundError(
            "Không tìm thấy folder keyframes hoặc file .tar trong /kaggle/input"
        )

    tar_path = tar_files[0]
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Tìm thấy TAR: {tar_path}")

    with tarfile.open(tar_path, mode="r:*") as archive:
        members = [
            member
            for member in archive.getmembers()
            if member.isfile()
            and "/keyframes/"
            in ("/" + member.name.replace("\\", "/").lstrip("./")).lower()
            and Path(member.name).suffix.lower() in IMAGE_SUFFIXES
        ]

        if not members:
            raise RuntimeError(f"Không tìm thấy ảnh keyframe trong {tar_path}")

        print(f"Đang giải nén {len(members):,} keyframe...")
        for member in tqdm(members, desc="Extract", unit="frame"):
            destination = EXTRACT_DIR / Path(member.name).name
            if destination.is_file() and destination.stat().st_size == member.size:
                continue

            source = archive.extractfile(member)
            if source is None:
                continue
            with source, destination.open("wb") as output:
                shutil.copyfileobj(source, output)

    return EXTRACT_DIR


def find_model() -> str:
    """Ưu tiên yolov8n.pt đã upload; nếu không có thì Ultralytics tự tải."""
    uploaded_models = sorted(KAGGLE_INPUT.glob(f"**/{MODEL_NAME}"))
    return str(uploaded_models[0]) if uploaded_models else MODEL_NAME


# =============================================================================
# TÁCH VIDEO_ID, SHOT_ID VÀ FRAMES_ID TỪ TÊN FILE
# =============================================================================

def parse_keyframe_name(image_path: Path) -> tuple[str, int, int]:
    match = FILENAME_PATTERN.fullmatch(image_path.stem)

    video_id = match.group("video_id").lower()
    shot_id = int(match.group("shot_id"))
    frames_id = int(match.group("frames_id"))
    return video_id, shot_id, frames_id


def record_key(record: dict) -> tuple[str, int, int]:
    return (
        record["video_id"],
        int(record["shot_id"]),
        int(record["frames_id"]),
    )


def checkpoint_path(worker_id: int) -> Path:
    return CHECKPOINT_PREFIX.with_name(
        f"{CHECKPOINT_PREFIX.name}_gpu{worker_id}.jsonl"
    )


def checkpoint_files() -> list[Path]:
    return sorted(
        CHECKPOINT_PREFIX.parent.glob(f"{CHECKPOINT_PREFIX.name}_gpu*.jsonl")
    )


def load_checkpoints() -> dict[tuple[str, int, int], dict]:
    records: dict[tuple[str, int, int], dict] = {}

    for path in checkpoint_files():
        with path.open("r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                try:
                    record = json.loads(line)
                    records[record_key(record)] = record
                except (json.JSONDecodeError, KeyError, TypeError, ValueError):
                    print(f"Bỏ qua checkpoint lỗi: {path.name}:{line_number}")
    return records


# =============================================================================
# CHẠY YOLO
# =============================================================================

keyframes_dir = find_keyframes_dir()
images = sorted(
    path
    for path in keyframes_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)

if not images:
    raise RuntimeError(f"Không có ảnh trong {keyframes_dir}")

# Kiểm tra toàn bộ tên file trước khi chạy model để tránh lỗi giữa chừng.
image_keys = {path: parse_keyframe_name(path) for path in images}
if len(set(image_keys.values())) != len(image_keys):
    raise RuntimeError("Có keyframe bị trùng cặp (video_id, shot_id, frames_id)")

existing = load_checkpoints()
pending = [path for path in images if image_keys[path] not in existing]

gpu_count = torch.cuda.device_count()
devices: list[int | str]
if gpu_count:
    devices = list(range(min(MAX_GPUS, gpu_count)))
else:
    devices = ["cpu"]

# Tải weight một lần trước khi hai thread cùng khởi tạo model.
model_source = find_model()
download_guard = YOLO(model_source)
del download_guard


def run_worker(
    worker_id: int,
    device: int | str,
    worker_images: list[Path],
) -> int:
    """Một worker sở hữu riêng một YOLO model và một checkpoint."""
    if not worker_images:
        return 0

    local_yolo = YOLO(model_source)
    output_path = checkpoint_path(worker_id)

    with output_path.open("a", encoding="utf-8") as checkpoint:
        progress = tqdm(
            total=len(worker_images),
            desc=f"GPU {device}" if device != "cpu" else "CPU",
            unit="frame",
            position=worker_id,
            leave=True,
        )

        for start in range(0, len(worker_images), BATCH_SIZE):
            batch_paths = worker_images[start : start + BATCH_SIZE]
            
            # Đã tối ưu cấu hình predict: sử dụng half=True cho GPU
            predictions = local_yolo.predict(
            source=[str(path) for path in batch_paths],
                conf=CONFIDENCE,
                imgsz=IMAGE_SIZE,
                device=device,
                quantize=16,
                end2end=True,
                max_det=100,
                rect=True,
                verbose=False,
            )

            for image_path, prediction in zip(batch_paths, predictions):
                if prediction.boxes is None:
                    class_ids: list[int] = []
                else:
                    class_ids = [
                        int(value)
                        for value in prediction.boxes.cls.cpu().tolist()
                    ]

                counts = Counter(
                    prediction.names[class_id] for class_id in class_ids
                )
                video_id, shot_id, frames_id = image_keys[image_path]

                record = {
                    "video_id": video_id,
                    "shot_id": shot_id,
                    "frames_id": frames_id,
                    "objects": dict(counts.most_common()),
                }
                checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")

            checkpoint.flush()
            # Lệnh os.fsync đã được gỡ bỏ tại đây
            progress.update(len(batch_paths))

        progress.close()

    return len(worker_images)

print("\n===== YOLO26x OBJECT DETECTION =====")
print(f"Keyframes  : {keyframes_dir}")
print(f"Tổng ảnh   : {len(images):,}")
print(f"Đã xử lý   : {len(existing):,}")
print(f"Cần xử lý  : {len(pending):,}")
if gpu_count:
    for device in devices:
        print(f"GPU {device}      : {torch.cuda.get_device_name(device)}")
    if len(devices) < MAX_GPUS:
        print(f"Cảnh báo: chỉ tìm thấy {len(devices)} GPU, tự chạy fallback.")
else:
    print("Cảnh báo: không có CUDA, đang chạy bằng CPU.")

# Chia xen kẽ để hai GPU nhận số lượng ảnh gần bằng nhau.
worker_splits = [pending[index::len(devices)] for index in range(len(devices))]

with ThreadPoolExecutor(max_workers=len(devices)) as executor:
    futures = [
        executor.submit(run_worker, worker_id, device, worker_splits[worker_id])
        for worker_id, device in enumerate(devices)
    ]
    for future in as_completed(futures):
        future.result()  # phát hiện và báo lỗi worker ngay lập tức


# =============================================================================
# XUẤT JSON CUỐI CÙNG
# =============================================================================

all_records = load_checkpoints()
results = [all_records[image_keys[path]] for path in images]
results.sort(
    key=lambda item: (item["video_id"], item["shot_id"], item["frames_id"])
)

with OUTPUT_JSON.open("w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

print("\n===== HOÀN TẤT =====")
print(f"Số kết quả : {len(results):,}")
print(f"Output JSON: {OUTPUT_JSON}")
print("Chọn Save Version để file xuất hiện trong Notebook Output của Kaggle.")

Đang kiểm tra/cập nhật Ultralytics...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Tìm thấy keyframes: /kaggle/input/datasets/trieu241007/keyframes-p1/keyframes

===== YOLO26x OBJECT DETECTION =====
Keyframes  : /kaggle/input/datasets/trieu241007/keyframes-p1/keyframes
Tổng ảnh   : 414,983
Đã xử lý   : 0
Cần xử lý  : 414,983
GPU 0      : Tesla T4
GPU 1      : Tesla T4


GPU 1:   0%|          | 0/207491 [00:00<?, ?frame/s]

GPU 0:   0%|          | 0/207492 [00:00<?, ?frame/s]


===== HOÀN TẤT =====
Số kết quả : 414,983
Output JSON: /kaggle/working/yolo_object_results_yolo26l.json
Chọn Save Version để file xuất hiện trong Notebook Output của Kaggle.
